In [117]:
import os 
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv
import json
import math

In [118]:
load_dotenv(override=True)

openai = OpenAI()

MODEL = "gpt-4.1-mini"

In [119]:
system_message = """You are a helpful financial assistant.

Your role is to help users manage their income by suggesting how much they should save, invest, and spend based on the salary or income they provide.

Provide simple, practical, and easy-to-understand financial suggestions.
Be polite, courteous, and concise.
Keep your responses within 2 sentences.
"""

In [120]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        tool_responses = handle_tool_call(assistant_message)

        messages.extend(tool_responses)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages
        )

    return response.choices[0].message.content

In [121]:
def handle_tool_call(message):
    tool_responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_income":
            arguments = json.loads(tool_call.function.arguments)

            salary = arguments.get("salary")

            income_details = get_income(salary)

            response = {
                "role": "tool",
                "content": income_details,
                "tool_call_id": tool_call.id
            }

            tool_responses.append(response)

    return tool_responses



In [122]:
def get_income(salary):
    salary = float(salary)

    spend = salary * 0.50
    invest = salary * 0.20
    tax = salary * 0.10
    save = salary * 0.20

    return (
        f"The user's monthly salary is {salary}. "
        f"Suggested spending: {spend}, "
        f"investments: {invest}, "
        f"tax: {tax}, "
        f"savings: {save}."
    )


In [123]:
income_function = {
    "name": "get_income",
    "description": "Calculate suggested spending, investments, taxes, and savings based on the user's salary.",
    "parameters": {
        "type": "object",
        "properties": {
            "salary": {
                "type": "number",
                "description": "The user's monthly salary."
            }
        },
        "required": ["salary"],
        "additionalProperties": False
    },
    "strict": True
}

In [124]:
tools = [{"type":"function", "function": income_function}]

In [125]:
tools

[{'type': 'function',
  'function': {'name': 'get_income',
   'description': "Calculate suggested spending, investments, taxes, and savings based on the user's salary.",
   'parameters': {'type': 'object',
    'properties': {'salary': {'type': 'number',
      'description': "The user's monthly salary."}},
    'required': ['salary'],
    'additionalProperties': False},
   'strict': True}}]

In [126]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.
